In [ ]:
import pandas as pd
import plotly.express as px

In [ ]:
#montar google drive
from google.colab import drive
drive.mount('/content/drive')

#definir la ruta
RUTA = "/content/drive/Mydrive/dataset_default.csv"

#leer el archivo CSV
import pandas as pd
df = pd.read_csv(RUTA)
df.head()

In [ ]:
!find /content/drive/MyDrive -name "dataset_default.csv"

In [ ]:
df['¿Default/No default?'].value_counts(normalize=True)
for col in df.columns[0:3]:
# Calcular valores minimo y maximo de la columna
  minimo = df[col].min()
  maximo = df[col].max()
  print(f'Maximo/minimo columna "{col}": {maximo:.1f}/{minimo:.1f}')

In [ ]:
fig = px.scatter_3d(df, x='puntaje de credito', y = 'ingresos mensuales(USD)', z= 'Monto del prestamo(USD)', color='¿Default/No default', size_max = 6, opacity = 0.5)
fig.show()

In [ ]:
def crear_in_out(datos):
  'obtiene las entradas y salidas del modelo a partir de los datos originales'
  X = datos.iloc[:, 3].to_numpy()
  return X, Y
  X, Y = crear_in_out(df)
  X
  Y

In [ ]:
from sklearn.model_selection import train_test_split

def crear_train_val_test(X,Y):
    '''Crear los sets de entrenamiento (70%), validación (15%)
    y prueba (15%)'''
    xtr, x_resto, ytr, y_resto = train_test_split(X, Y, test_size=0.3,
                                                  random_state=22)
    xvl, xts, yvl, yts = train_test_split(x_resto, y_resto, test_size=0.5,
                                          random_state=22)

    return xtr, ytr, xvl, yvl, xts, yts

# Ejecutar la función
x_train, y_train, x_val, y_val, x_test, y_test = crear_train_val_test(X,Y)

# Verificar
print(x_train.shape, y_train.shape)
print(x_val.shape, y_val.shape)
print(x_test.shape, y_test.shape)

In [ ]:
# Haremos escalamiento entre 0 y 1 (normalización)
# En otro tutorial veremos otras formas de escalamiento

from sklearn.preprocessing import MinMaxScaler

def escalar_datos(xtr, xvl, xts):
    '''Escalar datos de entrada (entrenamiento, validación y prueba)
    en el rango de 0 a 1'''
    scaler = MinMaxScaler()

    # Fit + transform sobre el set de entrenamiento
    xtr_s = scaler.fit_transform(xtr)

    # Transform sobre prueba y validación
    xvl_s = scaler.transform(xvl)
    xts_s = scaler.transform(xts)

    return xtr_s, xvl_s, xts_s

# Ejecutar la función
x_train_s, x_val_s, x_test_s = escalar_datos(x_train, x_val, x_test)

# Verificar
print(x_train_s.min(axis=0),x_train_s.max(axis=0))
print(x_val_s.min(axis=0),x_val_s.max(axis=0))
print(x_test_s.min(axis=0),x_test_s.max(axis=0))

In [ ]:
# Sin escalamiento
print('Desempeño SIN escalamiento de los datos de entrada: ')
tree = arbol_entrenar_evaluar(x_train, y_train, x_val, y_val)

# Con escalamiento
print('-'*40)
print('Desempeño CON escalamiento de los datos de entrada: ')
tree_s = arbol_entrenar_evaluar(x_train_s, y_train, x_val_s, y_val)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

def clasificadorKNN_entrenar_evaluar(x_tr, y_tr, x_vl, y_vl):

    knn = KNeighborsClassifier(n_neighbors=3)
    knn.fit(x_tr, y_tr)
    train_acc = knn.score(x_tr, y_tr)
    val_acc = knn.score(x_vl, y_vl)

    print(f'Exactitud entrenamiento/validación: {100*train_acc:.2f}% / {100*val_acc:.2f}%')

    return knn

In [ ]:
# Sin escalamiento
print('Desempeño SIN escalamiento de los datos de entrada: ')
knn = clasificadorKNN_entrenar_evaluar(x_train, y_train, x_val, y_val)

# Con escalamiento
print('-'*40)
print('Desempeño CON escalamiento de los datos de entrada: ')
knn_s = clasificadorKNN_entrenar_evaluar(x_train_s, y_train, x_val_s, y_val)

In [ ]:
import tensorflow as tf

def neurona_entrenar_validar(x_tr, y_tr, x_vl, y_vl):
    '''Crea y entrena una neurona artificial en Keras, con estas características:
    - Tamaño de la entrada: 3x1
    - Tamaño de la capa oculta: 2 neuronas
    - Tamaño de la salida: 1 neurona (default/no default)'''

    # Fijar semilla del generador aleatorio
    tf.keras.utils.set_random_seed(45)

    # Crear la neurona
    modelo = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(input_shape=(3,1)), # Entrada
        tf.keras.layers.Dense(2, activation='relu'), # Capa oculta
        tf.keras.layers.Dense(1, activation='sigmoid') # Salida
        ])

    # Compilar el modelo
    modelo.compile(optimizer='sgd',
                    loss='BinaryCrossentropy',
                    metrics = ['accuracy'])

    # Entrenarlo
    modelo.fit(x_tr, y_tr, epochs=5, verbose=0)

    # Validarlo
    train_loss, train_acc = modelo.evaluate(x_tr, y_tr, verbose=0)
    val_loss, val_acc = modelo.evaluate(x_vl, y_vl, verbose=0)

    # Imprimir resultado en pantalla
    print(f'Exactitud entrenamiento/validación: {100*train_acc:.2f}% / {100*val_acc:.2f}%')


    return modelo

In [ ]:
# Sin escalamiento
print('Desempeño SIN escalamiento de los datos de entrada: ')
red = neurona_entrenar_validar(x_train, y_train, x_val, y_val)

# Con escalamiento
print('-'*40)
print('Desempeño CON escalamiento de los datos de entrada: ')
red_s = neurona_entrenar_validar(x_train_s, y_train, x_val_s, y_val)